In [1]:
import pytest
import ipytest
from surface import *

In [2]:
ipytest.autoconfig()

In [4]:
%%ipytest

# --- 1. Grid Generation Tests ---

def test_surface_grid_dimensions():
    # 2x2 meter surface with 10x10 resolution = 100 points
    s = Surface(center=[0, 0, 0], dims=(2, 2), const_axis=2, resolution=(10, 10))
    
    assert s.r_surface.shape == (100, 3)
    # Patch area should be (2/10) * (2/10) = 0.04
    assert s.A == pytest.approx(0.04)
    assert s.Tx_elements.N == 100
    assert s.Rx_elements.N == 100

def test_surface_centering():
    # Surface centered at Z=5. All points should have Z=5.
    center_z = 5.0
    s = Surface(center=[0, 0, center_z], dims=(1, 1), const_axis=2, resolution=(5, 5))
    
    assert np.all(s.r_surface[:, 2] == center_z)
    # Check bounds: 1m dim centered at 0 should go from -0.5 to 0.5
    # Centers of the first/last patches will be slightly inset
    assert np.min(s.r_surface[:, 0]) > -0.5
    assert np.max(s.r_surface[:, 0]) < 0.5

def test_const_axis_rotation():
    # Test YZ plane (const_axis = 0)
    center = [10, 0, 0]
    s = Surface(center=center, dims=(2, 2), const_axis=0, resolution=(2, 2))
    # All X coordinates must be 10
    assert np.all(s.r_surface[:, 0] == 10)

# --- 2. Element Property Tests ---

def test_window_type_power_calculation():
    # 1. Force Constants to a known state for this test
    # If Constants is a class with class attributes, we override them:
    Constants.pd_peak = 1.0 
    
    # 2. Mock sun_power using a lambda
    # We use a wrapper to ensure we see what's happening
    SpectralPhysics.sun_power = lambda: 10.0
    
    # 3. Create a window (1x1 area)
    s = Surface(center=[0,0,0], dims=(1,1), const_axis=2, resolution=(1,1), type="window")
    
    # Debug print: If this fails again, look at these values in the output
    print(f"DEBUG: Area={s.A}, pd_peak={Constants.pd_peak}, sun_power={SpectralPhysics.sun_power()}")
    
    # Power P = pd_peak * Area * sun_power
    # Expected: 1.0 * 1.0 * 10.0 = 10.0
    assert s.P == pytest.approx(10.0), f"Value was {s.P}, expected 10.0"

def test_element_inheritance():
    # Verify that nT and nR are correctly passed to the child elements
    custom_nT = np.array([0, 1, 0])
    s = Surface(center=[0,0,0], dims=(1,1), const_axis=2, resolution=(2,2), nT=custom_nT)
    
    # Tx elements should have the custom normal
    assert np.array_equal(s.Tx_elements.n[0], custom_nT)
    # Rx elements should have default [0,0,1]
    assert np.array_equal(s.Rx_elements.n[0], [0, 0, 1])

def test_surface_patch_area_sum():
    # The sum of all patch areas should equal the total surface area
    dims = (3, 5)
    res = (7, 12)
    s = Surface(center=[0,0,0], dims=dims, const_axis=2, resolution=res)
    
    total_area = dims[0] * dims[1]
    calculated_total = s.A * s.r_surface.shape[0]
    assert calculated_total == pytest.approx(total_area)

......                                                                                       [100%]
6 passed in 0.02s
